In [1]:
import pandas as pd

path = "../data/01-bronze/Inventario Computadoras.csv"
df_raw = pd.read_csv(path, sep=";", encoding="utf-8")
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 540 entries, 0 to 539
Data columns (total 21 columns):
 #   Column                                 Non-Null Count  Dtype 
---  ------                                 --------------  ----- 
 0   Number                                 540 non-null    int64 
 1   Name                                   540 non-null    object
 2   Deployment State                       540 non-null    object
 3   Incident State                         540 non-null    object
 4   DynamicField_TIPOPC                    540 non-null    object
 5   DynamicField_FABRICANTE                540 non-null    object
 6   DynamicField_MODELOPC                  540 non-null    object
 7   DynamicField_NUMEROSERIE               540 non-null    object
 8   DynamicField_ENTIDAD                   540 non-null    object
 9   DynamicField_CONTRATO                  540 non-null    object
 10  DynamicField_ADENDUM                   540 non-null    object
 11  DynamicField_USUARI

In [2]:
df_raw = df_raw.rename(
    columns={
        'Number' : 'Numero',
        'Name' : 'Nombre',
        'Deployment State' : 'Estado Despliegue',
        'Incident State' : 'Estado Incidente',
        'DynamicField_TIPOPC' : 'Tipo',
        'DynamicField_FABRICANTE' : 'Fabricante',
        'DynamicField_MODELOPC' : 'Modelo',
        'DynamicField_NUMEROSERIE' : 'Numero Serie',
        'DynamicField_ENTIDAD' : 'Entidad',
        'DynamicField_CONTRATO' : 'Contrato',
        'DynamicField_ADENDUM' : 'Adendum',
        'DynamicField_USUARIO' : 'Usuario',
        'DynamicField_CAPACIDADMEMORIARAM' : 'Capacidad MemoriaRAM',
        'DynamicField_CAPACIDADDISCODURO' : 'Capacidad DiscoDuro',
        'DynamicField_PROCESADOR' : 'Procesador',
        'DynamicField_SOINSTALADO' : 'SO Instalado',
        'DynamicField_VERSIONBIOS' : 'Version BIOS',
        'DynamicField_FECHAPUBLICACIONBIOS' : 'Fecha Publicacion BIOS',
        'DynamicField_FECHAFINGARANTIA' : 'Fecha Fin Garantia',
        'DynamicField_FECHAULTIMAACTUALIZACION' : 'Fecha Ultima Actualizacion',
        'DynamicField_OBSERVACIONES' : 'Observaciones'
    }
)
## normalize column names
for column in df_raw.columns:
    if df_raw[column].isnull().any():
        df_raw[column] = df_raw[column].fillna("Sin novedades")
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 540 entries, 0 to 539
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Numero                      540 non-null    int64 
 1   Nombre                      540 non-null    object
 2   Estado Despliegue           540 non-null    object
 3   Estado Incidente            540 non-null    object
 4   Tipo                        540 non-null    object
 5   Fabricante                  540 non-null    object
 6   Modelo                      540 non-null    object
 7   Numero Serie                540 non-null    object
 8   Entidad                     540 non-null    object
 9   Contrato                    540 non-null    object
 10  Adendum                     540 non-null    object
 11  Usuario                     540 non-null    object
 12  Capacidad MemoriaRAM        540 non-null    object
 13  Capacidad DiscoDuro         540 non-null    object

In [3]:
## Normalize bios name column
## varias marcas, lenovo, hp, dell, asus, apple. distintas distribuciones de bios
import re
def parsear_bios_lenovo(valor):
    if pd.isna(valor):
        return None, None
    valor = valor.strip()
    
    # Formato: CODIGO(X.XX) o CODIGO (X.XX )
    m = re.match(r'^([A-Z0-9]+)\s*\(\s*([\d.]+)\s*\)$', valor)
    if m:
        return m.group(1), m.group(2)
    
    # Formato: CODIGO X.XX o CODIGO/X.XX
    #m = re.match(r'^([A-Z0-9]+)[\s/]([\d.]+)', valor)
    m = re.match(r'^([A-Z0-9]+)[\s/](\d+\.\d{1,2})\s*$', valor)

    if m:
        return m.group(1), m.group(2)
    
    # Solo código sin versión numérica
    return valor, None

In [4]:
df_raw_copy = df_raw.copy()
df_raw_copy.loc[df_raw['Fabricante']=='LENOVO',['Codigo_BIOS', 'Version_Num']] = pd.DataFrame(
    df_raw.loc[df_raw['Fabricante']=='LENOVO','Version BIOS'].apply(parsear_bios_lenovo), 
    index=df_raw.index
)
df_raw_copy[df_raw_copy['Fabricante'] == 'APPLE']


,Numero,Nombre,Estado Despliegue,Estado Incidente,Tipo,Fabricante,Modelo,Numero Serie,Entidad,Contrato,...,Capacidad DiscoDuro,Procesador,SO Instalado,Version BIOS,Fecha Publicacion BIOS,Fecha Fin Garantia,Fecha Ultima Actualizacion,Observaciones,Codigo_BIOS,Version_Num
41,10260000510,PORTATIL-KYV0R6H4XW,OPERATIVO,OPERATIVO,PORTATIL,APPLE,MACBOOK AIR,KYV0R6H4XW,PROMOTORES INMOBILIARIOS PRONOBIS S.A,BIS-CT-2025-0077,...,256 GB,CPU DE 10 NUCLEOS,MACOS,BUILD 24C2101,2026-04-03 00:00:00,2026-10-31 00:00:00,2026-04-20 00:00:00,SIN NOVEDADES,NaN,NaN


In [5]:
df_raw['Entidad'].unique()

array(['PRAXMED', 'PROMOTORES INMOBILIARIOS PRONOBIS S.A',
       'SENSE-ECUADOR S.A.', 'SISTEMAS MEDICOS SIME SIMETECUIDA S.A.',
       'PICHINCHA CORP S.A.', 'ECOLAB', 'HOJAVERDE CIA. LTDA.',
       'IDEAL ALAMBREC S A', 'EMPRESA COMERCIAL VELA LEIVA EMCOVELE S.A.',
       'PUNTONET S.A', 'DIGITAL INFORMATION DIGITINFO S.A.',
       'CONSTRUECUADOR S.A.', 'AsesoríaControl S.A',
       'CLUB UNIVERSIDAD CATOLICA', 'JABONERIA WILSON S. A.'],
      dtype=object)

In [6]:
df_raw_copy[['Codigo_BIOS', 'Version_Num']] = pd.DataFrame(
    df_raw_copy['Version BIOS'].apply(parsear_bios_lenovo).to_list(), 
    index=df_raw_copy.index
)
df_raw_copy[df_raw_copy['Entidad'] == 'PUNTONET S.A'][['Fabricante', 'Codigo_BIOS', 'Version_Num']]


,Fabricante,Codigo_BIOS,Version_Num
289,ASUS,CRARL579.0029,None
290,ASUS,CRARL579.0029,None
291,ASUS,CRARL579.0029,None
292,ASUS,CRARL579.0029,None
293,ASUS,CRARL579.0029,None
294,ASUS,CRARL579.0029,None
295,ASUS,CRARL579.0029,None
296,ASUS,CRARL579.0029,None
297,ASUS,CRARL579.0029,None


In [7]:
def parsear_bios_lenovo_prueba(valor):
    if pd.isna(valor):
        return None
    valor = valor.strip()
    
    # Formato: CODIGO(X.XX) o CODIGO (X.XX )
    m = re.match(r'^([A-Z0-9]+)\s*\(\s*([\d.]+)\s*\)$', valor)
    if m:
        return f'{m.group(1)} {m.group(2)}'
    
    # Formato: CODIGO X.XX o CODIGO/X.XX
    #m = re.match(r'^([A-Z0-9]+)[\s/]([\d.]+)', valor)
    m = re.match(r'^([A-Z0-9]+)[\s/](\d+\.\d{1,2})\s*$', valor)

    if m:
        return f'{m.group(1)} {m.group(2)}'
    
    # Solo código sin versión numérica
    return valor

In [9]:
df_raw_copy['Version BIOS'] = pd.DataFrame(
    df_raw_copy['Version BIOS'].apply(parsear_bios_lenovo_prueba).to_list(), 
    index=df_raw_copy.index
)
df_raw_copy[df_raw_copy['Entidad'] == 'PRAXMED'][['Fabricante', 'Codigo_BIOS', 'Version BIOS']]


,Fabricante,Codigo_BIOS,Version BIOS
0,LENOVO,R30ET35W,R30ET35W 1.09
1,LENOVO,R31ET57W,R31ET57W 1.57
2,LENOVO,R31ET57W,R31ET57W 1.57
3,LENOVO,R30ET35W,R30ET35W 1.09
4,LENOVO,R30ET35W,R30ET35W 1.09
356,LENOVO,R31ET57W,R31ET57W 1.57
357,LENOVO,R31ET57W,R31ET57W 1.57
358,LENOVO,R31ET57W,R31ET57W 1.57
359,LENOVO,R30ET35W,R30ET35W 1.09
360,LENOVO,R30ET35W,R30ET35W 1.09


In [ ]:
df_raw['Version BIOS'] = pd.DataFrame(
    df_raw['Version BIOS'].apply(parsear_bios_lenovo_prueba), 
    index=df_raw.index
)
df_raw['Version BIOS'].unique()

array(['R30ET35W 1.09', 'R31ET57W 1.57', 'N44ET37W 1.20', 'N47ET28W 1.17',
       '01.09.01 REV.A', 'RKCN25WW', 'N4JET23W 1.13', 'N43ET42W 1.25',
       'BUILD 24C2101', 'V7.5.5.19', 'N3XET65W 1.40', 'N3QET51W 1.51',
       'N3AET89W 1.54', 'N3PET32W 1.23', 'N3YET83W 1.48', 'N3FET46W 1.70',
       '01.05.01 REV.A', '01.12.00 REV.A', '01.11.00 REV.A',
       '02.25.00 REV.A', '01.23.00 REV.A', '01.22.01 REV.A',
       'R2HET45W 1.24', 'CRARL579.0029', 'R2LET38W 1.19', 'R2JET45W 1.22',
       'R2AET66W 1.41', 'M47KT3FA/1.0.0.63', 'F8CN59WW', 'R1SET63W 1.34',
       '01.12.01 Rev.A', 'R2KET38W 1.27', 'R25ET47W 1.28',
       'N3YET82W 1.47', 'R24ET51W 1.34', 'N3XET64W 1.39', 'N3MET27W 1.26',
       'R1XET61W 1.44', 'R30ET37W 1.11', 'R30ET38W 1.12', 'R2JET40W 1.17',
       'R2AET62W 1.37', 'R2AET64W 1.39', 'R2AET55W 1.30', 'R24ET48W 1.31',
       'R2AET51W 1.26', 'R2AET63W 1.38', 'R2AET56W 1.31', 'R2AET54W 1.29',
       'M4TKT3FA', 'M4TKT3DA', 'M4TKT41A'], dtype=object)